In [1]:
!pip install -q fastapi uvicorn groq faiss-cpu  sentence-transformers pyngrok  nest-asyncio kaggle pandas

In [2]:
# ==========================================
# CELL 2 — Download Dataset from Kaggle
# ==========================================
import os
from google.colab import userdata  # أو files


# ✅ الطريقة 2: لو مش حاطاها في secrets — هيطلب upload
from google.colab import files
files.upload()  # ارفعي kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download الداتاست
!kaggle datasets download -d shamimhasan8/ml-and-ds-coding-interview-question-bank
!unzip -q ml-and-ds-coding-interview-question-bank.zip -d /content/data
!ls /content/data

Saving kaggle.json to kaggle (3).json
Dataset URL: https://www.kaggle.com/datasets/shamimhasan8/ml-and-ds-coding-interview-question-bank
License(s): MIT
ml-and-ds-coding-interview-question-bank.zip: Skipping, found more recently modified local copy (use --force to force download)
replace /content/data/coding_interview_question_bank.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
coding_interview_question_bank.csv


In [3]:
# ==========================================
# CELL 3 — Load & Explore Data
# ==========================================
import pandas as pd
import os

# شوفي الملفات الموجودة
data_dir = "/content/data/"
files = os.listdir(data_dir)
print("📁 Files found:", files)

# حمّلي الـ CSV
# (غالباً اسمه interview_questions.csv أو similar)
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(f"{data_dir}/{csv_file}")

print(f"\n✅ Shape: {df.shape}")
print(f"\n📋 Columns:\n{df.columns.tolist()}")
print(f"\n🔍 Sample:\n{df.head(3)}")
print(f"\n📊 Dtypes:\n{df.dtypes}")

📁 Files found: ['coding_interview_question_bank.csv']

✅ Shape: (1000, 5)

📋 Columns:
['id', 'question', 'difficulty', 'category', 'date']

🔍 Sample:
   id                                           question difficulty  \
0   1  What is your favorite machine learning algorit...     Medium   
1   2     What is the bias-variance tradeoff? (Sample 2)     Medium   
2   3  What are the key differences between classific...     Medium   

           category        date  
0  Machine Learning  2025-05-09  
1  Machine Learning  2025-03-15  
2  Machine Learning  2025-06-28  

📊 Dtypes:
id             int64
question      object
difficulty    object
category      object
date          object
dtype: object


In [4]:
import json, os, re

QUESTION_COL   = "question"
DIFFICULTY_COL = "difficulty"
CATEGORY_COL   = "category"

df_clean = df.copy()
df_clean.columns = df_clean.columns.str.strip().str.lower()
df_clean = df_clean.dropna(subset=[QUESTION_COL, CATEGORY_COL, DIFFICULTY_COL])

df_clean[QUESTION_COL]   = df_clean[QUESTION_COL].astype(str).str.strip().str.lower()
df_clean[CATEGORY_COL]   = df_clean[CATEGORY_COL].astype(str).str.strip().str.lower()
df_clean[DIFFICULTY_COL] = df_clean[DIFFICULTY_COL].astype(str).str.strip().str.lower()

# ✅ شيلي (sample X) من السؤال
df_clean[QUESTION_COL] = df_clean[QUESTION_COL].apply(
    lambda x: re.sub(r'\s*\(sample\s*\d+\)', '', x).strip()
)

# ✅ شيلي الـ duplicates بعد كده
df_clean = df_clean.drop_duplicates(subset=[QUESTION_COL])

print(f"📊 بعد إزالة التكرار: {len(df_clean)} سؤال فريد")
print(f"\n📊 Categories:\n{df_clean[CATEGORY_COL].value_counts()}")
print(f"\n📊 Difficulties:\n{df_clean[DIFFICULTY_COL].value_counts()}")

# RAG Format
questions_data = []
for _, row in df_clean.iterrows():
    question   = row[QUESTION_COL]
    category   = row[CATEGORY_COL]
    difficulty = row[DIFFICULTY_COL]
    questions_data.append({
        "question":   question,
        "category":   category,
        "difficulty": difficulty,
        "text":       f"{question} {category} {difficulty}"
    })

os.makedirs("/content/app/data", exist_ok=True)
with open("/content/app/data/questions.json", "w") as f:
    json.dump({"questions": questions_data}, f, indent=2)

print(f"\n💾 Saved {len(questions_data)} unique questions ✅")
print("\n📌 Sample:")
print(json.dumps(questions_data[0], indent=2))

📊 بعد إزالة التكرار: 20 سؤال فريد

📊 Categories:
category
machine learning    14
data science         3
deep learning        3
Name: count, dtype: int64

📊 Difficulties:
difficulty
easy      8
medium    6
hard      6
Name: count, dtype: int64

💾 Saved 20 unique questions ✅

📌 Sample:
{
  "question": "what is your favorite machine learning algorithm and why?",
  "category": "machine learning",
  "difficulty": "medium",
  "text": "what is your favorite machine learning algorithm and why? machine learning medium"
}


In [5]:
import json
import random

class RAGService:
    def __init__(self, path="/content/app/data/questions.json"):
        with open(path) as f:
            data = json.load(f)
        self.questions = data["questions"]
        print(f"✅ RAG ready! {len(self.questions)} questions loaded 🎯")

    def get_interview_questions(
        self,
        category:   str  = None,
        difficulty: str  = None,
        k:          int  = 5,
        exclude:    list = None
    ) -> list[str]:

        exclude = exclude or []

        # ✅ فلتر مباشر + بنرجع "question" مش "text"
        pool = [
            q["question"] for q in self.questions
            if (not difficulty or q["difficulty"].lower() == difficulty.lower())
            and (not category  or q["category"].lower()  == category.lower())
            and q["question"] not in exclude
        ]

        # ✅ عشوائية حقيقية = مفيش تكرار
        random.shuffle(pool)
        selected = pool[:k]

        # لو مش لاقي كفاية → fallback بدون فلتر
        if len(selected) < k:
            print(f"⚠️ Found only {len(selected)}, relaxing filters...")
            fallback = [
                q["question"] for q in self.questions
                if q["question"] not in exclude
                and q["question"] not in selected
            ]
            random.shuffle(fallback)
            selected.extend(fallback[:k - len(selected)])

        return selected


# ── Test ──────────────────────────────────────────────────
rag = RAGService()

print("\n🎯 Easy ML questions:")
for q in rag.get_interview_questions(category="machine learning", difficulty="easy", k=3):
    print(f"  - {q}")

print("\n🎯 Hard Deep Learning questions:")
for q in rag.get_interview_questions(category="deep learning", difficulty="hard", k=3):
    print(f"  - {q}")

# ✅ تأكدي إن مفيش تكرار
print("\n✅ Duplication test:")
q1 = rag.get_interview_questions(category="data science", difficulty="medium", k=5)
q2 = rag.get_interview_questions(category="data science", difficulty="medium", k=5)
print(f"Session 1: {q1}")
print(f"Session 2: {q2}")
print(f"Same? {q1 == q2}")  # المفروض يطبع False 🎉

✅ RAG ready! 20 questions loaded 🎯

🎯 Easy ML questions:
  - how would you explain the concept of overfitting?
  - what is regularization and why is it important?
  - describe the process of feature selection.

🎯 Hard Deep Learning questions:
⚠️ Found only 2, relaxing filters...
  - describe the process of backpropagation.
  - what is the role of activation functions in deep learning?
  - explain cross-validation and its purpose.

✅ Duplication test:
⚠️ Found only 1, relaxing filters...
⚠️ Found only 1, relaxing filters...
Session 1: ['how would you approach data cleaning for a large dataset?', 'what is the role of activation functions in deep learning?', 'what is regularization and why is it important?', 'how would you optimize a machine learning model?', 'describe the process of feature selection.']
Session 2: ['how would you approach data cleaning for a large dataset?', 'what is your favorite machine learning algorithm and why?', 'how do you handle missing values in a dataset?', 'de

In [6]:
from groq import Groq
from google.colab import userdata
import re

class GroqService:
    def __init__(self):
        self.client = Groq(api_key="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")
        self.model  = "llama-3.1-8b-instant"

    def evaluate_answer_stream(
        self, job_role: str, question: str, answer: str
    ):
        """
        ✨ Streaming evaluation — بترجع كل كلمة فور ما تيجي
        (مفيش reference answer لأن الداتا مفيهاش — الـ LLM يقيّم من معرفته)
        """
        prompt = f"""You are an expert interviewer for {job_role} positions.

**Question:** {question}
**Candidate's Answer:** {answer}

Evaluate the candidate's answer using this EXACT format:

**SCORE:** [1-10]

**FEEDBACK:**
[2-3 sentences — specific, honest, constructive]

**STRENGTHS:**
- [strength 1]
- [strength 2]

**AREAS TO IMPROVE:**
- [weakness 1]
- [weakness 2]

**IDEAL ANSWER:**
[What a perfect answer looks like in 3-4 sentences]"""

        stream = self.client.chat.completions.create(
            model       = self.model,
            messages    = [{"role": "user", "content": prompt}],
            temperature = 0.4,
            max_tokens  = 700,
            stream      = True
        )

        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                yield delta

    def generate_extra_questions(self, job_role: str, count: int = 3) -> list[str]:
        """للـ roles اللي مش في الداتا"""
        prompt = f"""Generate {count} technical interview questions for {job_role}.
Return ONLY a numbered list, nothing else.
1."""
        resp = self.client.chat.completions.create(
            model    = self.model,
            messages = [{"role": "user", "content": prompt}],
            max_tokens = 300
        )
        text  = resp.choices[0].message.content
        lines = [l.strip() for l in text.split('\n') if l.strip() and l[0].isdigit()]
        return [l.split('.', 1)[1].strip() for l in lines][:count]

    @staticmethod
    def extract_score(text: str) -> int:
        m = re.search(r'\*\*SCORE:\*\*\s*(\d+)', text)
        return min(10, max(1, int(m.group(1)))) if m else 5


# ── Test ──────────────────────────────────────────────────
groq_svc = GroqService()

print("⚡ Streaming test:")
for chunk in groq_svc.evaluate_answer_stream(
    "Data Scientist",
    "What is the bias-variance tradeoff?",
    "It is the balance between model complexity and generalization."
):
    print(chunk, end="", flush=True)

⚡ Streaming test:
**SCORE:** 7

**FEEDBACK:**
The candidate has a good understanding of the concept, but their answer could be more precise and detailed. They touch on the idea of model complexity, but don't explicitly mention the relationship between bias and variance, which is a crucial aspect of the tradeoff.

**STRENGTHS:**
- The candidate shows a general understanding of the concept.
- They mention model complexity, which is a key component of the bias-variance tradeoff.

**AREAS TO IMPROVE:**
- The candidate could provide a more detailed explanation of the bias-variance tradeoff, including its implications for model selection and performance.
- They should explicitly define bias and variance, and explain how they relate to each other in the context of the tradeoff.

**IDEAL ANSWER:**
The bias-variance tradeoff is a fundamental concept in machine learning that describes the relationship between a model's ability to fit the training data (bias) and its ability to generalize to new,

In [7]:
import uuid
from datetime import datetime

sessions = {}

class InterviewSession:
    def __init__(self, session_id, category, difficulty, user_name, user_email, questions):
        self.session_id  = session_id
        self.category    = category
        self.difficulty  = difficulty
        self.user_name   = user_name
        self.user_email  = user_email
        self.questions   = questions
        self.current_idx = 0
        self.evaluations = []
        self.started_at  = datetime.utcnow().isoformat()

    @property
    def is_finished(self): return self.current_idx >= len(self.questions)

    @property
    def current_question(self):
        return self.questions[self.current_idx] if not self.is_finished else None

    def advance(self, eval_text: str):
        self.evaluations.append({
            "question":  self.current_question,
            "eval_text": eval_text,
            "score":     GroqService.extract_score(eval_text)
        })
        self.current_idx += 1

    def average_score(self):
        if not self.evaluations: return 0
        return round(sum(e["score"] for e in self.evaluations) / len(self.evaluations), 1)

    def get_final_report(self):
        avg = self.average_score()
        verdict = (
            "🏆 Excellent — Ready for the role!" if avg >= 8 else
            "✅ Good — Minor improvements needed" if avg >= 6 else
            "⚠️ Fair — More practice required"    if avg >= 4 else
            "❌ Needs substantial work"
        )
        return {
            "session_id":      self.session_id,
            "user_name":       self.user_name,
            "user_email":      self.user_email,
            "category":        self.category,
            "difficulty":      self.difficulty,
            "average_score":   avg,
            "verdict":         verdict,
            "total_questions": len(self.questions),
            "evaluations":     self.evaluations,
            "started_at":      self.started_at,
            "completed_at":    datetime.utcnow().isoformat()
        }


def create_session(
    user_name:  str,
    user_email: str,
    category:   str = "machine learning",
    difficulty: str = "medium",
    n_questions: int = 5
) -> InterviewSession:

    # ✅ جيبي أسئلة عشوائية مختلفة في كل session
    questions = rag.get_interview_questions(
        category=category,
        difficulty=difficulty,
        k=n_questions,
        exclude=[]
    )

    # لو RAG مش لاقي كفاية → Groq يكمّل
    if len(questions) < n_questions:
        print(f"🤖 RAG found {len(questions)}, generating rest with Groq...")
        extra = groq_svc.generate_extra_questions(
            category, difficulty, count=n_questions - len(questions)
        )
        questions.extend(extra)

    sid     = str(uuid.uuid4())[:8]
    session = InterviewSession(sid, category, difficulty, user_name, user_email, questions)
    sessions[sid] = session
    return session


print("✅ Session manager ready!")

✅ Session manager ready!


In [8]:
import httpx
from google.colab import userdata

class N8NService:
    def __init__(self):
        self.webhook_url = "https://monymonmon.app.n8n.cloud/webhook-test/363aebf7-5d47-4060-b05d-3339bd8acf9b"

    async def send_results(self, report: dict):
        if not self.webhook_url:
            print("⚠️ N8N_WEBHOOK_URL مش موجود في الـ secrets!")
            return
        async with httpx.AsyncClient() as client:
            try:
                response = await client.post(
                    self.webhook_url,
                    json    = report,
                    timeout = 15.0
                )
                print(f"✅ n8n responded: {response.status_code}")
            except Exception as e:
                print(f"❌ n8n error: {e}")

n8n_service = N8NService()
print("✅ N8N service ready!")

✅ N8N service ready!


In [9]:
!pip install -q pyngrok

In [10]:
import nest_asyncio
import uvicorn
import threading
import json
import asyncio
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="AI Interview Coach 🎯")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

# ── Schemas ───────────────────────────────────────────────
class StartReq(BaseModel):
    job_role:   str
    user_name:  str
    user_email: str
    difficulty: str = "Medium"

class AnswerReq(BaseModel):
    session_id: str
    answer:     str

# ── role → category map ───────────────────────────────────
ROLE_TO_CATEGORY = {
    "Data Scientist":  "data science",
    "ML Engineer":     "machine learning",
    "NLP Engineer":    "machine learning",
    "Data Analyst":    "data science",
    "AI Researcher":   "deep learning"
}

# ── Endpoints ─────────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "AI Interview Coach is live 🚀"}

@app.post("/interview/start")
def start(req: StartReq):
    category = ROLE_TO_CATEGORY.get(req.job_role, "machine learning")

    session = create_session(
        user_name   = req.user_name,
        user_email  = req.user_email,
        category    = category,
        difficulty  = req.difficulty.lower(),
        n_questions = 5
    )
    return {
        "session_id":      session.session_id,
        "question":        session.current_question,
        "question_number": 1,
        "total_questions": len(session.questions)
    }

@app.post("/interview/answer/stream")
def answer_stream(req: AnswerReq):
    session = sessions.get(req.session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    if session.is_finished:
        raise HTTPException(400, "Interview already finished")

    full_eval = []

    def generate():
        for chunk in groq_svc.evaluate_answer_stream(
            session.category,           # ✅ fix: كان session.job_role
            session.current_question,
            req.answer
        ):
            full_eval.append(chunk)
            yield f"data: {json.dumps({'type':'chunk','content':chunk})}\n\n"

        eval_text = "".join(full_eval)
        session.advance(eval_text)

        meta = {
            "type":            "done",
            "score":           GroqService.extract_score(eval_text),
            "is_finished":     session.is_finished,
            "next_question":   session.current_question,
            "question_number": session.current_idx + 1,
            "total_questions": len(session.questions),
            "average_so_far":  session.average_score()
        }

        if session.is_finished:
            report = session.get_final_report()
            meta["final_report"] = report
            asyncio.run(n8n_service.send_results(report))

        yield f"data: {json.dumps(meta)}\n\n"

    return StreamingResponse(
        generate(),
        media_type = "text/event-stream",
        headers    = {"Cache-Control": "no-cache", "X-Accel-Buffering": "no"}
    )

@app.get("/interview/session/{sid}")
def get_status(sid: str):
    s = sessions.get(sid)
    if not s:
        raise HTTPException(404, "Not found")
    return {
        "session_id":    sid,
        "category":      s.category,
        "difficulty":    s.difficulty,
        "progress":      f"{s.current_idx}/{len(s.questions)}",
        "average_score": s.average_score(),
        "is_finished":   s.is_finished
    }

# ── ngrok + uvicorn ───────────────────────────────────────
ngrok.set_auth_token("2vlPnT77TzLVXMyHSrOHGN4SwlM_7S6FbuW2xnY9VriPit4tV")
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print(f"\n🌐 Public API URL : {public_url}")
print(f"📖 Swagger docs   : {public_url}/docs\n")

import builtins
builtins.BACKEND_URL = public_url

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run, daemon=True).start()

import time; time.sleep(2)
print("✅ FastAPI is running!")


🌐 Public API URL : https://76f4-34-143-184-173.ngrok-free.app
📖 Swagger docs   : https://76f4-34-143-184-173.ngrok-free.app/docs

✅ FastAPI is running!


In [11]:
streamlit_code = f"""
import streamlit as st
import requests
import json
import time

BACKEND = "{public_url}/interview"

st.set_page_config(
    page_title="AI Interview Coach",
    page_icon="🎯",
    layout="centered"
)

st.title("🎯 AI Interview Coach")
st.caption("Powered by Groq + RAG Interview System")

defaults = {{
    "session_id": None,
    "question":   None,
    "q_num":      0,
    "total":      0,
    "done":       False,
    "report":     None,
    "history":    []
}}

for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

if not st.session_state.session_id:

    st.subheader("👋 Start Interview")

    name  = st.text_input("Name")
    email = st.text_input("Email")

    job_role = st.selectbox("Role", [
        "Data Scientist",
        "ML Engineer",
        "NLP Engineer",
        "Data Analyst",
        "AI Researcher"
    ])

    difficulty = st.select_slider(
        "Difficulty",
        ["Easy", "Medium", "Hard"],
        value="Medium"
    )

    if st.button("🚀 Start"):
        if name and email:
            try:
                response = requests.post(
                    f"{{BACKEND}}/start",
                    json={{
                        "job_role":   job_role,
                        "user_name":  name,
                        "user_email": email,
                        "difficulty": difficulty
                    }},
                    timeout=15
                )
                if response.status_code != 200:
                    st.error(f"❌ Server error {{response.status_code}}: {{response.text}}")
                    st.stop()
                res = response.json()
            except requests.exceptions.ConnectionError:
                st.error("❌ Can't reach backend! Is FastAPI running?")
                st.stop()
            except requests.exceptions.JSONDecodeError:
                st.error(f"❌ Bad response: {{response.text[:300]}}")
                st.stop()

            st.session_state.session_id = res["session_id"]
            st.session_state.question   = res["question"]
            st.session_state.q_num      = 1
            st.session_state.total      = res["total_questions"]
            st.rerun()
        else:
            st.warning("Enter name + email")

elif not st.session_state.done:

    st.progress(
        (st.session_state.q_num - 1) / st.session_state.total,
        text=f"Q {{st.session_state.q_num}} / {{st.session_state.total}}"
    )

    st.subheader(st.session_state.question)

    answer = st.text_area("Your answer", height=150)

    if st.button("Submit"):
        if not answer.strip():
            st.warning("Write answer first")
        else:
            st.markdown("### 🤖 AI Evaluation")
            placeholder = st.empty()
            eval_text   = ""

            with requests.post(
                f"{{BACKEND}}/answer/stream",
                json={{
                    "session_id": st.session_state.session_id,
                    "answer":     answer
                }},
                stream=True
            ) as resp:

                for line in resp.iter_lines():
                    if line and line.startswith(b"data: "):
                        data = json.loads(line[6:])

                        if data.get("type") == "chunk":
                            eval_text += data["content"]
                            placeholder.markdown(eval_text + "▌")

                        elif data.get("type") == "done":
                            placeholder.markdown(eval_text)

                            score = data.get("score", 5)

                            if score >= 7:
                                color = "green"
                            elif score >= 4:
                                color = "orange"
                            else:
                                color = "red"

                            st.markdown(
                                f"<h3 style='color:{{color}}'>Score: {{score}}/10</h3>",
                                unsafe_allow_html=True
                            )

                            st.session_state.history.append({{
                                "question": st.session_state.question,
                                "score":    score
                            }})

                            if data.get("is_finished"):
                                st.session_state.done   = True
                                st.session_state.report = data.get("final_report")
                            else:
                                st.session_state.question = data.get("next_question")
                                st.session_state.q_num    = data.get("question_number")

                            time.sleep(1)
                            st.rerun()

else:

    r = st.session_state.report

    st.balloons()
    st.success("🎉 Completed!")

    col1, col2, col3 = st.columns(3)
    col1.metric("Avg Score", f"{{r['average_score']}}/10")
    col2.metric("Questions", r["total_questions"])
    col3.metric("Result",    r["verdict"])

    st.markdown("### 📊 Report")

    for i, ev in enumerate(r["evaluations"], 1):
        q_text  = ev["question"]
        q_score = ev["score"]
        q_eval  = ev["eval_text"]
        st.markdown(f"### Q{{i}}")
        st.markdown("**Question:** " + q_text)
        st.markdown("**Score:** " + str(q_score))
        st.markdown(q_eval)
        st.divider()

    if st.button("Restart"):
        for k in list(st.session_state.keys()):
            del st.session_state[k]
        st.rerun()
"""

with open("/content/streamlit_app.py", "w") as f:
    f.write(streamlit_code)

print("✅ Streamlit saved!")

✅ Streamlit saved!


In [12]:
!pip install -q streamlit pyngrok

In [13]:
import subprocess, time
from pyngrok import ngrok

proc = subprocess.Popen([
    "streamlit", "run", "/content/streamlit_app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false"
])

time.sleep(4)

streamlit_url = ngrok.connect(8501).public_url
print(f"\n🎨 Streamlit UI  : {streamlit_url}")
print(f"🌐 FastAPI Docs  : {public_url}/docs")
print("\n👆 افتحي الـ Streamlit URL في المتصفح وابدأي! 🎯")


🎨 Streamlit UI  : https://d592-34-143-184-173.ngrok-free.app
🌐 FastAPI Docs  : https://76f4-34-143-184-173.ngrok-free.app/docs

👆 افتحي الـ Streamlit URL في المتصفح وابدأي! 🎯
